In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



/kaggle/input/prompt-injection-dataset/MPDD.csv
/kaggle/input/datasets/kamrun71/final-models/linearsvc.joblib
/kaggle/input/datasets/kamrun71/final-models/xgboost.joblib
/kaggle/input/datasets/kamrun71/final-models/random_forest.joblib
/kaggle/input/datasets/kamrun71/final-models/logistic_regression.joblib
/kaggle/input/datasets/kamrun71/processed-data/y_train.npy
/kaggle/input/datasets/kamrun71/processed-data/y_test.npy
/kaggle/input/datasets/kamrun71/processed-data/y_val.npy
/kaggle/input/datasets/kamrun71/processed-data/X_test_vec.npy
/kaggle/input/datasets/kamrun71/processed-data/X_val_vec.npy
/kaggle/input/datasets/kamrun71/processed-data/X_train_vec.npy
/kaggle/input/datasets/kamrun71/feature-3/feature_combiner_3.pkl
/kaggle/input/datasets/kamrun71/ensemble-models/ensemble_full_features.pkl
/kaggle/input/datasets/kamrun71/ensemble-models/cross_ensemble_lr_full_svc_xgb_opt.pkl
/kaggle/input/datasets/kamrun71/ensemble-models/cross_ensemble_lr_xgb.pkl
/kaggle/input/datasets/kamrun71

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score
import re
import nltk
from nltk.corpus import stopwords
# Ensure have the 'stopwords' and 'punkt' resources downloaded
# nltk.download('stopwords')
# nltk.download('punkt')

# Load the dataset
# Replace 'MPDD.csv' with the actual path if it's not in the same directory
df = pd.read_csv('/kaggle/input/prompt-injection-dataset/MPDD.csv')
# Remove case-insensitive duplicates while keeping the first occurrence


# Check new dataset size
print(len(df))


# Display the first few rows and check the class distribution
print("Data Head:")
print(df.head())
print("\nClass Distribution:")
print(df['isMalicious'].value_counts())

39234
Data Head:
                                              Prompt  isMalicious
0  How did the "Coronation Street" storyline hand...            1
1                                 Login to Snapchat?            0
2  I've been having a hard time getting back into...            1
3           What is the best way to budget my money?            1
4  Is it rude to say that I don’t understand “Cau...            0

Class Distribution:
isMalicious
1    19617
0    19617
Name: count, dtype: int64


In [3]:
# Check null prompts
null_count = df["Prompt"].isna().sum()
print("Null prompts:", null_count)

# Check empty / whitespace-only prompts
empty_count = df["Prompt"].astype(str).str.strip().eq("").sum()
print("Empty or whitespace-only prompts:", empty_count)

# Check exact duplicates
duplicate_exact = df.duplicated(subset="Prompt").sum()
print("Exact duplicate prompts:", duplicate_exact)

# Check case-insensitive duplicates
duplicate_ci = df["Prompt"].str.lower().duplicated().sum()
print("Case-insensitive duplicate prompts:", duplicate_ci)

Null prompts: 0
Empty or whitespace-only prompts: 0
Exact duplicate prompts: 0
Case-insensitive duplicate prompts: 1


In [4]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import re
import nltk
from nltk.corpus import stopwords
!pip install langdetect  # Run this in a cell if not already installed

from langdetect import detect, DetectorFactory
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb

# Deduplication
# ============================
# DATASET-LEVEL CLEANING
# (OUTSIDE CV – correct)
# ============================

# Deduplication
df = df.loc[~df["Prompt"].str.lower().duplicated()]

# English filtering
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 0

def is_english(text):
    try:
        return detect(str(text)) == "en"
    except:
        return False

df = df[df["Prompt"].apply(is_english)].copy()

print("Dataset size after dedup + English filter:", len(df))
print(df["isMalicious"].value_counts())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 14.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=e59bd6c4b533375e4d345e4cb821ec6a84a5e1e7197581de4e1d3ba60c44688e
  Stored in directory: /root/.cache/pip/wheels/0a/f2/b2/e5ca405801e05eb7c8ed5b3b4bcf1fcabcd6272c167640072e
Successfully built langdetect
Dataset size after dedup + English filter: 37547
isMalicious
0    19013
1    18534
Name: count, dtype: int64


In [5]:
import re
import numpy as np
import pandas as pd
import string
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

import xgboost as xgb
import torch
from transformers import AutoTokenizer, AutoModel

# -----------------------------
# 1. INJECTION-AWARE PREPROCESSING
# -----------------------------
def injection_aware_preprocess(text):
    if pd.isna(text) or text is None:
        return ""
    text = text.lower()
    text = re.sub(r'([^\w\s])', r' \1 ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def preprocess_wrapper(X):
    return X['Prompt'].apply(injection_aware_preprocess)

# -----------------------------
# -----------------------------
# 2. ADVERSARIAL FEATURES (EXACT MASTER CHECK IMPLEMENTATION)
# -----------------------------

# Ensure Spacy is loaded for Layer 5
try:
    import spacy
    nlp_engine = spacy.load("en_core_web_sm")
except OSError:
    print("WARNING: Spacy model 'en_core_web_sm' not found. Layer 5 features will be 0.")
    nlp_engine = None

class AdversarialFeatureExtractor(BaseEstimator, TransformerMixin):

    def __init__(self):
        # We define the keywords exactly as you found them
        self.keyword_groups = {
            "instruction_override": ["ignore", "forget", "override", "bypass", "previous", "instruction", "prompt", "format", "generate"],
            "role_manipulation": ["system", "assistant", "developer", "admin", "role", "you are"],
            "sql_like_payload": ["select", "update", "delete", "insert", "drop", "where"],
            "technical_context": ["report", "economy", "blame", "regex", "expression", "python", "java", "code"],
            "style_and_art": ["style", "ghibli", "sorting"],
            # Flatten the rest for a general "Suspicious Keyword" check
            "general_suspicious": [
                "start", "following", "regarding", "business", "news", "class", "name", 
                "when", "if", "prior", "answer", "question", "disregard", "what", "how", 
                "good", "way", "skip", "overlook", "neglect", "minorities", "hate"
            ]
        }
        # Multi-word phrases from Layer 2
        self.phrases = ["is it possible", "can you", "could you", "you are"]

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        series = X['Prompt'].astype(str)
        
        # Pre-calculate lowercase once for speed
        series_lower = series.str.lower()
        
        features = pd.DataFrame()

        # ==============================================================================
        # LAYER 1: Technical Escape & Syntax (Your Findings)
        # ==============================================================================
        # 1. Check for the specific "Full-width Comma" evasion
        features['flag_fullwidth_comma'] = series.apply(lambda t: 1 if '，' in t else 0)
        # 2. Check for Code Arrow "->"
        features['flag_code_arrow'] = series.apply(lambda t: 1 if '->' in t else 0)
        # 3. Check for Escaped Newlines "\ n" or "\\n"
        features['flag_escaped_newline'] = series.apply(lambda t: 1 if '\\n' in t or '\\ n' in t.lower() else 0)

        # ==============================================================================
        # LAYER 2 & 3: Phrases & Keywords
        # ==============================================================================
        # Check for specific phrase attacks
        features['flag_phrase_injection'] = series_lower.apply(
            lambda t: 1 if any(p in t for p in self.phrases) else 0
        )

        # Keyword Groups (mapped to features)
        features['flag_kw_override'] = series_lower.apply(
            lambda t: 1 if any(k in t for k in self.keyword_groups["instruction_override"]) else 0
        )
        features['flag_kw_role'] = series_lower.apply(
            lambda t: 1 if any(k in t for k in self.keyword_groups["role_manipulation"]) else 0
        )
        features['flag_kw_sql'] = series_lower.apply(
            lambda t: 1 if any(k in t for k in self.keyword_groups["sql_like_payload"]) else 0
        )
        features['flag_kw_tech'] = series_lower.apply(
            lambda t: 1 if any(k in t for k in self.keyword_groups["technical_context"]) else 0
        )
        features['flag_kw_style'] = series_lower.apply(
            lambda t: 1 if any(k in t for k in self.keyword_groups["style_and_art"]) else 0
        )
        
        # Catch-all for any other keyword from your list
        features['flag_kw_general'] = series_lower.apply(
            lambda t: 1 if any(k in t for k in self.keyword_groups["general_suspicious"]) else 0
        )

        # ==============================================================================
        # LAYER 4: Punctuation Structure
        # ==============================================================================
        def check_punctuation_layer(text):
            if not text: return 0
            last_char = text.strip()[-1] if text.strip() else ""
            # Checks standard OR the specific unicode symbols you found
            if last_char in string.punctuation or last_char in ["。", "！", "？", "，"]:
                return 1
            return 0
            
        features['flag_punctuation_check'] = series.apply(check_punctuation_layer)

        # ==============================================================================
        # LAYER 5: Linguistic Structure (Spacy)
        # ==============================================================================
        # This implements your check: "Is there a ROOT VERB or a Modal (MD)?"
        def check_linguistics(text):
            if nlp_engine is None: return 0
            try:
                # We limit text length for speed, as injection headers are usually at the start
                doc = nlp_engine(text[:512]) 
                if any(t.pos_ == "VERB" and t.dep_ == "ROOT" for t in doc) or \
                   any(t.tag_ == "MD" for t in doc):
                    return 1
            except:
                return 0
            return 0

        # Note: This step is slower than regex, but necessary for exact pattern matching
        features['flag_linguistic_imperative'] = series_lower.apply(check_linguistics)

        # ==============================================================================
        # Extra Density Stats (Helpful for ML decision boundaries)
        # ==============================================================================
        features['count_parentheses'] = series.apply(lambda t: t.count('(') + t.count(')'))
        features['char_density'] = series.apply(
            lambda t: len(re.findall(r'[^\w\s]', t)) / len(t) if t else 0
        )

        return features.values

    def get_feature_names_out(self, input_features=None):
        return np.array([
            'flag_fullwidth_comma', 
            'flag_code_arrow', 
            'flag_escaped_newline',
            'flag_phrase_injection',
            'flag_kw_override',
            'flag_kw_role',
            'flag_kw_sql',
            'flag_kw_tech',
            'flag_kw_style',
            'flag_kw_general',
            'flag_punctuation_check',
            'flag_linguistic_imperative',
            'count_parentheses',
            'char_density'
        ])


# -----------------------------
# 3. BERT EMBEDDINGS (OPTIMIZED BATCHING)
# -----------------------------
MODEL_NAME = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(DEVICE)
model.eval()
BERT_DIM = model.config.hidden_size

class BERTFeatureExtractor(BaseEstimator, TransformerMixin):

    def __init__(self, tokenizer, model, device, max_len=128, batch_size=32):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
        self.max_len = max_len
        self.batch_size = batch_size  # New parameter for speed

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Handle cases where X is a DataFrame or Series
        if isinstance(X, pd.DataFrame):
            series = X['Prompt'].tolist()
        else:
            series = X.tolist()

        embeddings = []
        total = len(series)
        
        print(f"Processing {total} samples on {self.device} with batch size {self.batch_size}...")

        # --- BATCH PROCESSING LOOP ---
        # This is 10x-50x faster than the "for text in series" loop
        for i in range(0, total, self.batch_size):
            # 1. Create a batch of texts
            batch_texts = series[i : i + self.batch_size]
            
            # Clean inputs (handle NaNs)
            batch_texts = [str(t) if pd.notnull(t) else "" for t in batch_texts]

           # 1. Determine the ACTUAL device the model is currently using
            current_model_device = next(self.model.parameters()).device

            # 2. Tokenize the WHOLE batch at once and send to THAT device
            inputs = self.tokenizer(
                batch_texts,
                return_tensors="pt",
                max_length=self.max_len,
                truncation=True,
                padding=True 
            ).to(current_model_device) # <--- THIS IS THE FIX

            # 3. Inference
            with torch.no_grad():
                outputs = self.model(**inputs)

            # 4. Extract CLS token for the whole batch
            cls_vectors = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            
            embeddings.append(cls_vectors)

            # Optional: Print progress every 1000 samples
            if i % 1000 == 0 and i > 0:
                print(f"  > Processed {i}/{total}...")

        # Stack all batches into one large numpy array
        return np.vstack(embeddings)


# -----------------------------
# 4. FEATURE PIPELINES
# -----------------------------
preprocess_step = FunctionTransformer(preprocess_wrapper)

word_tfidf = Pipeline([
    ('clean', preprocess_step),
    ('tfidf', TfidfVectorizer(analyzer='word', ngram_range=(1, 3), max_features=4000))
])

char_tfidf = Pipeline([
    ('clean', preprocess_step),
    ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=3000))
])

adv_pipeline = Pipeline([
    ('adv', AdversarialFeatureExtractor())
])

# Update  pipeline to use the batch_size
bert_pipeline = Pipeline([
    ('bert', BERTFeatureExtractor(tokenizer, model, DEVICE, batch_size=64)) # Try 64 for T4 GPU
])


# -----------------------------
# 5. COMBINED FEATURE DESIGN
# -----------------------------
feature_combiner = ColumnTransformer(
    transformers=[
        ('word_tfidf', word_tfidf, ['Prompt']),
        ('char_tfidf', char_tfidf, ['Prompt']),
        ('adv_features', adv_pipeline, ['Prompt']),
        ('bert_embed', bert_pipeline, ['Prompt'])
    ],
    remainder='drop'
)

print("Hybrid + BERT Feature Engineering Ready!")

# -----------------------------
# 6. LOAD + SPLIT (NO TRAINING)
# -----------------------------


# Check new dataset size
print("Samples after removing case-insensitive duplicates:", len(df))
X = df[['Prompt']]
y = df['isMalicious']

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

print("Train:", len(X_train), "| Val:", len(X_val), "| Test:", len(X_test))


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

...

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Hybrid + BERT Feature Engineering Ready!
Samples after removing case-insensitive duplicates: 37547
Train: 22527 | Val: 7510 | Test: 7510


In [6]:
"""
=============================================================
  COMPUTATIONAL COST ANALYSIS — BASE + RETRAINED MODELS ONLY
  Device: CPU | Runs: 10,000 per model
  Ensemble already done separately — excluded here.
  Measures:
    - Inference only (ms/sample)
    - Total latency (feature extraction + inference)
=============================================================
"""

import time
import psutil
import joblib
import pandas as pd
import numpy as np
import scipy.sparse as sp
import os
import torch
from unittest.mock import patch
from sklearn.metrics import accuracy_score

# =============================================================
# 1. HELPERS
# =============================================================

def to_numpy(data):
    if torch.is_tensor(data):
        return data.cpu().numpy().astype(np.float32)
    elif sp.issparse(data):
        return data.toarray().astype(np.float32)
    return np.array(data, dtype=np.float32)

original_torch_load = torch.load

def cpu_torch_load(f, *args, **kwargs):
    kwargs['map_location'] = torch.device('cpu')
    kwargs.setdefault('weights_only', False)
    return original_torch_load(f, *args, **kwargs)

def load_npy(path):
    arr = np.load(path, allow_pickle=True)
    if arr.ndim == 0:
        arr = arr.item()
    return arr

NUM_RUNS = 10000
device   = "cpu"

# =============================================================
# 2. LOAD DATA
# =============================================================
print("=" * 60)
print("  STEP 1: LOAD DATA")
print("=" * 60)

y_test = np.array(load_npy(
    "/kaggle/input/datasets/kamrun71/processed-data/y_test.npy"
))
N_test = len(y_test)
print(f"  Test samples : {N_test}")

# =============================================================
# 3. LOAD FEATURE COMBINER + INDICES
# =============================================================
print("\n" + "=" * 60)
print("  STEP 2: LOAD FEATURE COMBINER + INDICES")
print("=" * 60)

start = time.time()
with patch('torch.load', cpu_torch_load):
    feature_combiner = joblib.load(
        "/kaggle/input/datasets/kamrun71/feature-3/feature_combiner_3.pkl"
    )
import_time_full = (time.time() - start) * 1000
print(f"  Combiner loaded : {import_time_full:.1f} ms")

start = time.time()
keep_indices = joblib.load(
    "/kaggle/input/datasets/kamrun71/retrained-models/final_feature_indices.pkl"
)
import_time_opt = (time.time() - start) * 1000
print(f"  Indices loaded  : {import_time_opt:.1f} ms")

# =============================================================
# 4. LOAD BASE + RETRAINED MODELS ONLY
# =============================================================
print("\n" + "=" * 60)
print("  STEP 3: LOAD MODELS")
print("=" * 60)

orig_dir = "/kaggle/input/datasets/kamrun71/final-models"
ret_dir  = "/kaggle/input/datasets/kamrun71/retrained-models"

base_lr  = joblib.load(f"{orig_dir}/logistic_regression.joblib")
base_rf  = joblib.load(f"{orig_dir}/random_forest.joblib")
base_svc = joblib.load(f"{orig_dir}/linearsvc.joblib")
base_xgb = joblib.load(f"{orig_dir}/xgboost.joblib")

ret_lr   = joblib.load(f"{ret_dir}/retrained_logistic_regression.pkl")
ret_rf   = joblib.load(f"{ret_dir}/retrained_random_forest.pkl")
ret_svc  = joblib.load(f"{ret_dir}/retrained_linearsvc.pkl")
ret_xgb  = joblib.load(f"{ret_dir}/retrained_xgboost.pkl")

print("  All models loaded.")

# =============================================================
# 5. FEATURE EXTRACTION — runs once, reused by all models
# =============================================================
print("\n" + "=" * 60)
print("  STEP 4: FEATURE EXTRACTION (runs once)")
print("=" * 60)

print(f"  Extracting test features ({N_test} samples)...")

# Full extraction
start = time.time()
with patch('torch.load', cpu_torch_load):
    X_test_full = feature_combiner.transform(X_test)
t_extract_full_per = (time.time() - start) * 1000 / N_test
X_test_full = to_numpy(X_test_full)

# Opt slicing
start = time.time()
X_test_opt = X_test_full[:, keep_indices]
t_extract_opt_per = (time.time() - start) * 1000 / N_test

t_extract_combined_per = t_extract_full_per + t_extract_opt_per

print(f"  Full extraction  : {t_extract_full_per:.6f} ms/sample")
print(f"  Opt slicing      : {t_extract_opt_per:.6f} ms/sample")
print(f"  Combined         : {t_extract_combined_per:.6f} ms/sample")
print(f"  Full shape       : {X_test_full.shape}")
print(f"  Opt shape        : {X_test_opt.shape}")

# System stats — captured once
cpu_usage = psutil.cpu_percent(interval=1)
process   = psutil.Process(os.getpid())
memory_mb = process.memory_info().rss / (1024 * 1024)
print(f"\n  CPU usage : {cpu_usage}%")
print(f"  Memory    : {memory_mb:.1f} MB")

# =============================================================
# 6. BENCHMARK FUNCTION
# =============================================================

def benchmark_single(model_name, model, X, y_true,
                     feat_extract_per, is_svc=False):
    """
    10,000 inference runs.
    Returns inference only AND total latency separately.
    """
    inf_times = []

    for i in range(NUM_RUNS):
        start = time.time()
        preds = model.predict(X)
        inf_ms = (time.time() - start) * 1000
        inf_times.append(inf_ms / N_test)

        if i % 2000 == 0 and i > 0:
            print(f"    > {i}/{NUM_RUNS} done...")

    inf_per = np.mean(inf_times)
    inf_std = np.std(inf_times)
    acc     = accuracy_score(y_true, preds)
    total   = feat_extract_per + inf_per

    print(f"  {model_name:<45} | "
          f"Inf: {inf_per:.6f} ms/sample | "
          f"Total: {total:.6f} ms/sample | "
          f"Acc: {acc:.4f}")

    return {
        "Stage"                          : "",
        "Model"                          : model_name,
        "Device"                         : device,
        "Inference Only (ms/sample)"     : round(inf_per, 6),
        "Inference Std (ms/sample)"      : round(inf_std, 6),
        "Feature Extraction (ms/sample)" : round(feat_extract_per, 6),
        "Total Latency (ms/sample)"      : round(total, 6),
        "CPU %"                          : round(cpu_usage, 1),
        "Memory (MB)"                    : round(memory_mb, 2),
        "Test Accuracy"                  : round(acc, 4),
    }

# =============================================================
# 7. RUN BENCHMARK — BASE MODELS
#    Uses: X_test_full (7782 features)
#    Feature cost: full extraction only
# =============================================================
print("\n" + "=" * 60)
print("  GROUP 1: BASE MODELS")
print("  Features : Full (7782)")
print("  Feat cost: full combiner only")
print("=" * 60)

all_results = []

for name, model, is_svc in [
    ("Logistic Regression", base_lr,  False),
    ("Random Forest",       base_rf,  False),
    ("Linear SVC",          base_svc, True),
    ("XGBoost",             base_xgb, False),
]:
    print(f"\n  Benchmarking {name}...")
    row = benchmark_single(
        model_name       = name,
        model            = model,
        X                = X_test_full,
        y_true           = y_test,
        feat_extract_per = t_extract_full_per,
        is_svc           = is_svc,
    )
    row["Stage"] = "Base"
    all_results.append(row)

# =============================================================
# 8. RUN BENCHMARK — RETRAINED MODELS
#    Uses: X_test_opt (2896 features)
#    Feature cost: full extraction + opt slicing
# =============================================================
print("\n" + "=" * 60)
print("  GROUP 2: RETRAINED MODELS")
print("  Features : Optimized (2896)")
print("  Feat cost: full combiner + opt slicing")
print("=" * 60)

for name, model, is_svc in [
    ("Logistic Regression (Retrained)", ret_lr,  False),
    ("Random Forest (Retrained)",       ret_rf,  False),
    ("Linear SVC (Retrained)",          ret_svc, True),
    ("XGBoost (Retrained)",             ret_xgb, False),
]:
    print(f"\n  Benchmarking {name}...")
    row = benchmark_single(
        model_name       = name,
        model            = model,
        X                = X_test_opt,
        y_true           = y_test,
        feat_extract_per = t_extract_combined_per,
        is_svc           = is_svc,
    )
    row["Stage"] = "Retrained"
    all_results.append(row)

# =============================================================
# 9. PRINT RESULTS
# =============================================================
print("\n\n" + "=" * 60)
print("  FINAL COMPUTATIONAL COST RESULTS")
print("=" * 60)

df = pd.DataFrame(all_results)

display_cols = [
    "Stage", "Model",
    "Inference Only (ms/sample)", "Inference Std (ms/sample)",
    "Feature Extraction (ms/sample)",
    "Total Latency (ms/sample)",
    "Test Accuracy",
    "CPU %", "Memory (MB)",
]

print(df[display_cols].to_string(index=False))

# =============================================================
# 10. FASTEST PER STAGE
# =============================================================
print("\n\n" + "=" * 60)
print("  FASTEST MODEL PER STAGE (by Total Latency)")
print("=" * 60)
for stage in ["Base", "Retrained"]:
    subset  = df[df["Stage"] == stage]
    fastest = subset.loc[subset["Total Latency (ms/sample)"].idxmin()]
    print(f"  {stage:<12} → {fastest['Model']:<45} "
          f"| {fastest['Total Latency (ms/sample)']:.6f} ms/sample "
          f"| Acc: {fastest['Test Accuracy']:.4f}")

# =============================================================
# 11. SAVE
# =============================================================
df[display_cols].to_csv("computational_cost_base_retrained.csv", index=False)
df.to_csv("computational_cost_base_retrained_full.csv", index=False)

print("\n\n  Saved → computational_cost_base_retrained.csv")
print("  Saved → computational_cost_base_retrained_full.csv")
print("\n  DONE.")

  STEP 1: LOAD DATA
  Test samples : 7510

  STEP 2: LOAD FEATURE COMBINER + INDICES
  Combiner loaded : 15236.2 ms
  Indices loaded  : 9.7 ms

  STEP 3: LOAD MODELS
  All models loaded.

  STEP 4: FEATURE EXTRACTION (runs once)
  Extracting test features (7510 samples)...
Processing 7510 samples on cuda with batch size 64...
  Full extraction  : 120.129839 ms/sample
  Opt slicing      : 0.042270 ms/sample
  Combined         : 120.172108 ms/sample
  Full shape       : (7510, 7782)
  Opt shape        : (7510, 2896)

  CPU usage : 0.3%
  Memory    : 8029.1 MB

  GROUP 1: BASE MODELS
  Features : Full (7782)
  Feat cost: full combiner only

  Benchmarking Logistic Regression...
    > 2000/10000 done...
    > 4000/10000 done...
    > 6000/10000 done...
    > 8000/10000 done...
  Logistic Regression                           | Inf: 0.014231 ms/sample | Total: 120.144070 ms/sample | Acc: 0.9715

  Benchmarking Random Forest...
    > 2000/10000 done...
    > 4000/10000 done...
    > 6000/1000